In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from tqdm import tqdm

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016
numpc = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  
predictions = []
y_test_list = []
dates = []
dic_r2_all = {}
numpc_time = {}

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    mse = np.full((len(numpc), 1), np.nan, dtype=np.float32)

    for i in range(len(numpc)):
        pca_val = PCA(n_components=numpc[i])
        X_reduced_train = pca_val.fit_transform(X_train)
        X_reduced_val = pca_val.transform(X_val)
        line_fitter_val = LinearRegression()
        line_fitter_val.fit(X_reduced_train, y_train)
        Yval_predict = line_fitter_val.predict(X_reduced_val)
        mse[i, 0] = np.sqrt(mean_squared_error(y_val, Yval_predict)) 
    optim_numpc = numpc[np.argmin(mse)]
    pca = PCA(n_components=optim_numpc)
    principalComponents = pca.fit_transform(np.concatenate((X_train, X_val)))
    X_reduced_test = pca.transform(X_test)    
    line_fitter = LinearRegression()
    line_fitter.fit(principalComponents, (np.concatenate((y_train, y_val))))
    preds = line_fitter.predict(X_reduced_test) 
    predictions.append(preds)
    dates.append(test_data['DATE'].values)
    y_test_list.append(y_test)
    r2 = 1 - np.sum((y_test - preds) ** 2) / np.sum(y_test ** 2)
    dic_r2_all[f"r2.{year}"] = r2
    numpc_time[f"numpc.{year}"] = optim_numpc


predictions_all = np.concatenate(predictions, axis=0)
y_test_list_all = np.concatenate(y_test_list, axis=0)
dates_all = np.concatenate(dates, axis=0)
R2OOS_PCR = 1 - np.sum((y_test_list_all - predictions_all) ** 2) / np.sum(y_test_list_all ** 2)
print("R2OOS principal component regression: ", R2OOS_PCR)
df_intermediate = pd.DataFrame({
    'Date': dates_all,
    'Prediction': predictions_all,
    'True Value': y_test_list_all
})

for year in range(start_year_test, end_year_test + 1):
    df_intermediate[f'r2.{year}'] = dic_r2_all[f"r2.{year}"]
    df_intermediate[f'numpc.{year}'] = numpc_time[f"numpc.{year}"]

print(df_intermediate.head())